In [11]:
# Installing Pyspark
%pip install -q pyspark

In [12]:
# starting and verifying Spark
import sys
import pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

print("Python version:", sys.version.split()[0])
print("PySpark version:", pyspark.__version__)

spark = (
    SparkSession.builder
    .appName("DATA226_SelfHealingPipeline")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark session started successfully!")

Python version: 3.13.18
PySpark version: 4.0.4
Spark version: 4.0.4
Spark session started successfully!


In [13]:
# Downloading and inspecting the GTFS dataset
import requests
import os
import zipfile
import io

GTFS_URL = "https://gtfs.vta.org/gtfs_vta.zip"
GTFS_PATH = "/content/gtfs_vta.zip"

response = requests.get(GTFS_URL, timeout=60)
response.raise_for_status()

with open(GTFS_PATH, "wb") as f:
    f.write(response.content)

print("Download successful!")
print("File size (MB):", round(len(response.content) / (1024**2), 2))
print("Saved to:", GTFS_PATH)

with zipfile.ZipFile(GTFS_PATH, "r") as z:
    files = z.namelist()

print("\nFiles in GTFS archive:")
for name in files:
    print(name)

Download successful!
File size (MB): 4.49
Saved to: /content/gtfs_vta.zip

Files in GTFS archive:
agency.txt
__MACOSX/._agency.txt
calendar_attributes.txt
__MACOSX/._calendar_attributes.txt
calendar_dates.txt
__MACOSX/._calendar_dates.txt
calendar.txt
__MACOSX/._calendar.txt
directions.txt
__MACOSX/._directions.txt
fare_attributes.txt
__MACOSX/._fare_attributes.txt
fare_rules.txt
__MACOSX/._fare_rules.txt
feed_info.txt
__MACOSX/._feed_info.txt
levels.txt
__MACOSX/._levels.txt
pathways.txt
__MACOSX/._pathways.txt
realtime_routes.txt
__MACOSX/._realtime_routes.txt
route_attributes.txt
__MACOSX/._route_attributes.txt
routes.txt
__MACOSX/._routes.txt
shapes.txt
__MACOSX/._shapes.txt
stop_times.txt
__MACOSX/._stop_times.txt
stops.txt
__MACOSX/._stops.txt
trips.txt
__MACOSX/._trips.txt


In [14]:
# Extracting the GTFS files in Google Colab
import os
import zipfile

# Location of the downloaded GTFS archive
GTFS_PATH = "/content/gtfs_vta.zip"

# Folder where the selected files will be extracted
DATA_DIR = "/content/gtfs_data"

os.makedirs(DATA_DIR, exist_ok=True)

# Four primary GTFS files for our initial experiment
required_files = [
    "routes.txt",
    "stops.txt",
    "trips.txt",
    "stop_times.txt"
]

# Extract and verify the required files
with zipfile.ZipFile(GTFS_PATH, "r") as z:
    available_files = set(z.namelist())

    for filename in required_files:
        if filename not in available_files:
            raise FileNotFoundError(
                f"{filename} was not found in the GTFS archive"
            )

        with z.open(filename) as source:
            output_path = os.path.join(DATA_DIR, filename)

            with open(output_path, "wb") as destination:
                destination.write(source.read())

print("GTFS files extracted successfully!")
print("\nExtracted files:")

for filename in required_files:
    path = os.path.join(DATA_DIR, filename)
    file_size = os.path.getsize(path)

    print(f"{filename}: {file_size:,} bytes")

print("\nExtraction directory:", DATA_DIR)


GTFS files extracted successfully!

Extracted files:
routes.txt: 8,501 bytes
stops.txt: 248,947 bytes
trips.txt: 485,030 bytes
stop_times.txt: 19,716,399 bytes

Extraction directory: /content/gtfs_data


In [15]:
# measuring actual row counts and inspect the schemas
from pyspark.sql import functions as F
import os

tables = {}

for filename in required_files:
    table_name = filename.replace(".txt", "")
    file_path = os.path.join(DATA_DIR, filename)

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("mode", "FAILFAST")
        .csv(file_path)
    )

    tables[table_name] = df

    print("\n" + "=" * 50)
    print("Table:", table_name)
    print("Columns:", len(df.columns))
    print("Rows:", df.count())
    print("Schema:")
    df.printSchema()

print("\nAll four GTFS tables loaded successfully into Spark.")



Table: routes
Columns: 11
Rows: 71
Schema:
root
 |-- route_id: string (nullable = true)
 |-- agency_id: string (nullable = true)
 |-- route_short_name: string (nullable = true)
 |-- route_long_name: string (nullable = true)
 |-- route_desc: string (nullable = true)
 |-- route_type: string (nullable = true)
 |-- route_url: string (nullable = true)
 |-- route_color: string (nullable = true)
 |-- route_text_color: string (nullable = true)
 |-- route_sort_order: string (nullable = true)
 |-- ext_route_type: string (nullable = true)


Table: stops
Columns: 14
Rows: 3345
Schema:
root
 |-- stop_id: string (nullable = true)
 |-- stop_code: string (nullable = true)
 |-- stop_name: string (nullable = true)
 |-- stop_desc: string (nullable = true)
 |-- stop_lat: string (nullable = true)
 |-- stop_lon: string (nullable = true)
 |-- zone_id: string (nullable = true)
 |-- stop_url: string (nullable = true)
 |-- location_type: string (nullable = true)
 |-- parent_station: string (nullable = true)
 |

In [16]:
# Checking all 10 columns of stop_times for null or blank values.

from pyspark.sql import functions as F

# Profile missing and blank values in the stop_times table
stop_times = tables["stop_times"]

missing_exprs = [
    F.sum(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in stop_times.columns
]

missing_profile = stop_times.agg(*missing_exprs)

print("Missing and blank values by column in stop_times:")
display(missing_profile.toPandas())


Missing and blank values by column in stop_times:


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint
0,0,0,0,0,0,387798,0,0,8620,0


In [17]:
# Building the PO baseline pipeline

import os

BASELINE_DIR = "/content/pipeline_outputs/p0_baseline"

os.makedirs(os.path.dirname(BASELINE_DIR), exist_ok=True)

# P0: preserve the original data without repair rules
(
    tables["stop_times"]
    .write
    .mode("overwrite")
    .parquet(BASELINE_DIR)
)

# Read the saved output back to verify it
baseline_df = spark.read.parquet(BASELINE_DIR)

source_count = tables["stop_times"].count()
baseline_count = baseline_df.count()

print("P0 baseline pipeline completed.")
print("Source rows:", source_count)
print("Parquet rows:", baseline_count)
print("Row counts match:", source_count == baseline_count)
print("Output directory:", BASELINE_DIR)


P0 baseline pipeline completed.
Source rows: 387798
Parquet rows: 387798
Row counts match: True
Output directory: /content/pipeline_outputs/p0_baseline


In [18]:
# Controlled Fault Injection
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load the original stop_times table
source_df = tables["stop_times"]

# Create a row ID for the controlled experiment
row_window = Window.orderBy(
    "trip_id", "stop_sequence", "stop_id"
)

indexed_df = source_df.withColumn(
    "experiment_row_id",
    F.row_number().over(row_window)
)

# Assign the intended fault type to three records
fault_trip_id = indexed_df.withColumn(
    "fault_type",
    F.when(
        F.col("experiment_row_id") == 1,
        "missing_trip_id"
    ).when(
        F.col("experiment_row_id") == 2,
        "invalid_arrival_time"
    ).when(
        F.col("experiment_row_id") == 3,
        "duplicate_record"
    )
)

print("Rows selected for controlled fault injection:")

fault_trip_id.filter(
    F.col("experiment_row_id").isin([1, 2, 3])
).select(
    "experiment_row_id",
    "trip_id",
    "arrival_time",
    "stop_id",
    "stop_sequence",
    "fault_type"
).show(truncate=False)


Rows selected for controlled fault injection:
+-----------------+-------+------------+-------+-------------+--------------------+
|experiment_row_id|trip_id|arrival_time|stop_id|stop_sequence|fault_type          |
+-----------------+-------+------------+-------+-------------+--------------------+
|1                |4020335|06:13:00    |1757   |1            |missing_trip_id     |
|2                |4020335|06:43:52    |653    |10           |invalid_arrival_time|
|3                |4020335|06:46:00    |1599   |11           |duplicate_record    |
+-----------------+-------+------------+-------+-------------+--------------------+



In [19]:
# creating a seperate fault injection dataframe
from pyspark.sql import functions as F

# Create a separate dataset for the fault-injection experiment.
# The original tables["stop_times"] and P0 baseline remain unchanged.

faulty_base = indexed_df

# Inject fault 1: missing trip_id in row 1
faulty_base = faulty_base.withColumn(
    "trip_id",
    F.when(
        F.col("experiment_row_id") == 1,
        F.lit(None).cast("string")
    ).otherwise(F.col("trip_id"))
)

# Inject fault 2: invalid arrival_time in row 2
faulty_base = faulty_base.withColumn(
    "arrival_time",
    F.when(
        F.col("experiment_row_id") == 2,
        F.lit("invalid_time")
    ).otherwise(F.col("arrival_time"))
)

# Inject fault 3: duplicate row 3
duplicate_row = (
    faulty_base
    .filter(F.col("experiment_row_id") == 3)
    .withColumn("experiment_row_id", F.lit(387799))
)

faulty_data = faulty_base.unionByName(duplicate_row)

print("Fault injection completed.")
print("Original row count:", indexed_df.count())
print("Fault-injected row count:", faulty_data.count())

print("\nInjected fault examples:")
faulty_data.filter(
    F.col("experiment_row_id").isin([1, 2, 3, 387799])
).select(
    "experiment_row_id",
    "trip_id",
    "arrival_time",
    "stop_id",
    "stop_sequence"
).show(truncate=False)


Fault injection completed.
Original row count: 387798
Fault-injected row count: 387799

Injected fault examples:
+-----------------+-------+------------+-------+-------------+
|experiment_row_id|trip_id|arrival_time|stop_id|stop_sequence|
+-----------------+-------+------------+-------+-------------+
|1                |NULL   |06:13:00    |1757   |1            |
|2                |4020335|invalid_time|653    |10           |
|3                |4020335|06:46:00    |1599   |11           |
|387799           |4020335|06:46:00    |1599   |11           |
+-----------------+-------+------------+-------+-------------+



In [20]:
# Building First Detection stage

from pyspark.sql import functions as F

# Detect missing trip identifiers
missing_trip = faulty_data.filter(
    F.col("trip_id").isNull() |
    (F.trim(F.col("trip_id")) == "")
).withColumn(
    "detected_fault", F.lit("missing_trip_id")
)

# Detect arrival times that do not match H:MM:SS or HH:MM:SS
valid_time_pattern = r"^\d{1,2}:\d{2}:\d{2}$"

invalid_arrival = faulty_data.filter(
    F.col("arrival_time").isNull() |
    (~F.col("arrival_time").rlike(valid_time_pattern))
).withColumn(
    "detected_fault", F.lit("invalid_arrival_time")
)

# Detect exact duplicate records using the original GTFS columns.
# Exclude the experiment ID, because duplicates have different experiment IDs.
gtfs_columns = tables["stop_times"].columns

duplicate_groups = (
    faulty_data
    .groupBy(*gtfs_columns)
    .count()
    .filter(F.col("count") > 1)
)

duplicates = faulty_data.join(
    duplicate_groups.select(*gtfs_columns),
    on=gtfs_columns,
    how="inner"
).withColumn(
    "detected_fault", F.lit("duplicate_record")
)

print("Missing trip_id detections:", missing_trip.count())
print("Invalid arrival_time detections:", invalid_arrival.count())
print("Rows belonging to duplicate groups:", duplicates.count())

print("\nDetected missing trip_id:")
missing_trip.select(
    "experiment_row_id", "trip_id", "arrival_time"
).show(truncate=False)

print("Detected invalid arrival_time:")
invalid_arrival.select(
    "experiment_row_id", "trip_id", "arrival_time"
).show(truncate=False)

print("Detected duplicate records:")
duplicates.select(
    "experiment_row_id", "trip_id", "arrival_time",
    "stop_id", "stop_sequence"
).show(truncate=False)


Missing trip_id detections: 1
Invalid arrival_time detections: 1
Rows belonging to duplicate groups: 0

Detected missing trip_id:
+-----------------+-------+------------+
|experiment_row_id|trip_id|arrival_time|
+-----------------+-------+------------+
|1                |NULL   |06:13:00    |
+-----------------+-------+------------+

Detected invalid arrival_time:
+-----------------+-------+------------+
|experiment_row_id|trip_id|arrival_time|
+-----------------+-------+------------+
|2                |4020335|invalid_time|
+-----------------+-------+------------+

Detected duplicate records:
+-----------------+-------+------------+-------+-------------+
|experiment_row_id|trip_id|arrival_time|stop_id|stop_sequence|
+-----------------+-------+------------+-------+-------------+
+-----------------+-------+------------+-------+-------------+



In [21]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Use the original GTFS columns, excluding experiment_row_id
gtfs_columns = tables["stop_times"].columns

# Count identical GTFS records, including rows with null fields
duplicate_window = Window.partitionBy(*gtfs_columns)

duplicate_check = faulty_data.withColumn(
    "duplicate_count",
    F.count("*").over(duplicate_window)
)

duplicates = (
    duplicate_check
    .filter(F.col("duplicate_count") > 1)
    .withColumn(
        "detected_fault",
        F.lit("duplicate_record")
    )
)

print("Rows belonging to duplicate groups:", duplicates.count())

duplicates.select(
    "experiment_row_id",
    "trip_id",
    "arrival_time",
    "stop_id",
    "stop_sequence",
    "duplicate_count"
).show(truncate=False)


Rows belonging to duplicate groups: 2
+-----------------+-------+------------+-------+-------------+---------------+
|experiment_row_id|trip_id|arrival_time|stop_id|stop_sequence|duplicate_count|
+-----------------+-------+------------+-------+-------------+---------------+
|3                |4020335|06:46:00    |1599   |11           |2              |
|387799           |4020335|06:46:00    |1599   |11           |2              |
+-----------------+-------+------------+-------+-------------+---------------+



In [22]:
# data recovery and quarantine

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Original GTFS columns (excluding experiment_row_id)
gtfs_columns = tables["stop_times"].columns

# Rule 1: Identify records with missing trip IDs
missing_trip_condition = (
    F.col("trip_id").isNull() |
    (F.trim(F.col("trip_id")) == "")
)

# Rule 2: Identify invalid arrival times
valid_time_pattern = r"^\d{1,2}:\d{2}:\d{2}$"

invalid_time_condition = (
    F.col("arrival_time").isNull() |
    (~F.col("arrival_time").rlike(valid_time_pattern))
)

# Quarantine records that fail either validation rule
quarantine_df = faulty_data.filter(
    missing_trip_condition | invalid_time_condition
)

# Keep records that pass both validation rules
valid_df = faulty_data.filter(
    ~missing_trip_condition & ~invalid_time_condition
)

# Rule 3: Remove exact duplicate GTFS records, keeping one copy
dedup_window = Window.partitionBy(*gtfs_columns).orderBy(
    F.col("experiment_row_id")
)

recovered_df = (
    valid_df
    .withColumn(
        "dedup_rank",
        F.row_number().over(dedup_window)
    )
    .filter(F.col("dedup_rank") == 1)
    .drop("dedup_rank", "experiment_row_id")
)

# Remove experiment-only IDs from quarantine output too
quarantine_output = quarantine_df.drop("experiment_row_id")

print("Recovery stage completed.")
print("Original source rows:", tables["stop_times"].count())
print("Fault-injected rows:", faulty_data.count())
print("Quarantined rows:", quarantine_output.count())
print("Recovered rows:", recovered_df.count())

print("\nQuarantined records:")
quarantine_output.select(
    "trip_id", "arrival_time", "stop_id", "stop_sequence"
).show(truncate=False)


Recovery stage completed.
Original source rows: 387798
Fault-injected rows: 387799
Quarantined rows: 2
Recovered rows: 387796

Quarantined records:
+-------+------------+-------+-------------+
|trip_id|arrival_time|stop_id|stop_sequence|
+-------+------------+-------+-------------+
|NULL   |06:13:00    |1757   |1            |
|4020335|invalid_time|653    |10           |
+-------+------------+-------+-------------+



In [25]:
# Evaluating detection performance

# Evaluate the three controlled fault events
true_faults = 3

detected_missing_trip = missing_trip.count()
detected_invalid_time = invalid_arrival.count()

detected_duplicate_groups = (
    duplicate_check
    .filter(F.col("duplicate_count") > 1)
    .select(*gtfs_columns)
    .distinct()
    .count()
)

# Confirm that each injected fault was detected
tp_missing = int(
    missing_trip.filter(
        F.col("experiment_row_id") == 1
    ).count() > 0
)

tp_invalid = int(
    invalid_arrival.filter(
        F.col("experiment_row_id") == 2
    ).count() > 0
)

tp_duplicate = int(
    duplicate_check
    .filter(
        (F.col("duplicate_count") > 1) &
        (F.col("experiment_row_id").isin([3, 387799]))
    )
    .select(*gtfs_columns)
    .distinct()
    .count() > 0
)

true_positives = tp_missing + tp_invalid + tp_duplicate

predicted_events = (
    detected_missing_trip
    + detected_invalid_time
    + detected_duplicate_groups
)

false_positives = predicted_events - true_positives
false_negatives = true_faults - true_positives

precision = (
    true_positives / (true_positives + false_positives)
    if true_positives + false_positives > 0 else 0
)

recall = (
    true_positives / (true_positives + false_negatives)
    if true_positives + false_negatives > 0 else 0
)

print("Known injected fault events:", true_faults)
print("Predicted fault events:", predicted_events)
print("True positives:", true_positives)
print("False positives:", false_positives)
print("False negatives:", false_negatives)
print(f"Detection precision: {precision:.2%}")
print(f"Detection recall: {recall:.2%}")

Known injected fault events: 3
Predicted fault events: 3
True positives: 3
False positives: 0
False negatives: 0
Detection precision: 100.00%
Detection recall: 100.00%


In [24]:
# Saving the recovered data and quarantine records

import os

OUTPUT_DIR = "/content/pipeline_outputs"

RECOVERED_PATH = os.path.join(
    OUTPUT_DIR, "p2_recovered"
)

QUARANTINE_PATH = os.path.join(
    OUTPUT_DIR, "p2_quarantine"
)

# Save the recovered dataset
recovered_df.write.mode("overwrite").parquet(
    RECOVERED_PATH
)

# Save the quarantined records
quarantine_output.write.mode("overwrite").parquet(
    QUARANTINE_PATH
)

# Read back and verify the saved outputs
saved_recovered = spark.read.parquet(RECOVERED_PATH)
saved_quarantine = spark.read.parquet(QUARANTINE_PATH)

print("Recovery outputs saved successfully.")
print("Saved recovered rows:", saved_recovered.count())
print("Saved quarantine rows:", saved_quarantine.count())
print("Recovered output:", RECOVERED_PATH)
print("Quarantine output:", QUARANTINE_PATH)



Recovery outputs saved successfully.
Saved recovered rows: 387796
Saved quarantine rows: 2
Recovered output: /content/pipeline_outputs/p2_recovered
Quarantine output: /content/pipeline_outputs/p2_quarantine
